# Peak shaving vs TCIPC analysis

Let's check the results. First we need to copy the results from the cluster to
this PC:
```
rsync -avm --include='*/' --include='*.sql*' --exclude='*' jhummel@login.delftblue.tudelft.nl:../../scratch/jhummel/tip_clearance/data/optimal_tuning/ ./data/optimal_tuning/ --dry-run
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.interpolate import griddata
from weis.visualization.utils import load_OMsql_multi, load_OMsql

plt.style.use("journal.mplstyle")

%matplotlib widget

In [ ]:
# Define the logs to load and give them labels.
logs_to_load = {
    "Free yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_free_yaw.sql",
    "Zero yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_zero_yaw.sql",
}

# Load all datasets
all_data_dicts = {}
for log_name, log_fmt in logs_to_load.items():
    all_data_dicts[log_name] = load_OMsql(log_fmt)
    print(f"Loaded {log_name}: {all_data_dicts[log_name].keys()}")

In [ ]:
# Let's define how we load, scale, and label the data, then make a dataframe.
all_outputs = {
    # ROSCO variables.
    "TCIPC_MaxTipDeflection": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC Max Tip Deflection (m)",
    },
    "ps_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: x[0],
        "label": "Peak Shaving (-)",
    },
    "TCIPC_nHarmonics": {
        "key": "tune_rosco_ivc.TCIPC_nHarmonics",
        "scaling": lambda x: x[0],
        "label": "Number of harmonics",
    },
    "TCIPC_ZeroYawDeflection": {
        "key": "tune_rosco_ivc.TCIPC_ZeroYawDeflection",
        "scaling": lambda x: x[0],
        "label": "Zero yaw deflection",
    },
    # Objectives / responses
    "aep": {
        "key": "aeroelastic.AEP",
        "scaling": lambda x: 1e-6 * x[0],
        "label": "AEP (GWh)",
    },
    "max_TipDxc_towerPassing": {
        "key": "aeroelastic.max_TipDxc_towerPassing",
        "scaling": lambda x: x[0],
        "label": "Max TipDxc Tower Passing (m)",
    },
    "tower_clearance": {
        "key": "aeroelastic.max_TipDxc_towerPassing",
        "scaling": lambda x: 30 - x[0],
        "label": "Tower clearance (m)",
    },
    "TCIPC_amplitude_at_max_deflection": {
        "key": "aeroelastic.TCIPC_amplitude_at_max_deflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC amplitude (deg)",
    },
    "avg_pitch_travel": {
        "key": "aeroelastic.avg_pitch_travel",
        "scaling": lambda x: x[0],
        "label": "Avg Pitch Travel (deg)",
    },
    "DEL_RootMyb": {
        "key": "aeroelastic.DEL_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "DEL Root Myb (MNm)",
    },
    "max_TwrBsMyt": {
        "key": "aeroelastic.max_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Max Tower Base Myt (MNm)",
    },
}

# Build dataframe from mapping for each log
labels = {short: info["label"] for short, info in all_outputs.items()}
all_dfs = []

for log_name, data_dict in all_data_dicts.items():
    df_dict = {}
    for short_label, info in all_outputs.items():
        data = data_dict[info["key"]]
        scaled_data = list(map(info["scaling"], data))
        df_dict[short_label] = scaled_data

    df_temp = pd.DataFrame(df_dict)
    df_temp["log_name"] = log_name  # Add identifier column
    all_dfs.append(df_temp)

# Combine all dataframes
df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}")
df.head()

## Data exploration

In [ ]:
# Make a plot of the distribution of our design variables.
plt.figure()
sns.scatterplot(
    df,
    x="TCIPC_MaxTipDeflection",
    y="ps_percent",
    style="log_name",
)
plt.show()

In [ ]:
# Plot several outputs/objectives as a function of the design variables.
# Output variables to plot.
outputs = [
    "aep",
    "tower_clearance",
    "avg_pitch_travel",
    "TCIPC_amplitude_at_max_deflection",
    "DEL_RootMyb",
    "max_TwrBsMyt",
]

fig, axs = plt.subplots(len(logs_to_load), len(outputs), figsize=(12, 5))

for i, log_name in enumerate(logs_to_load.keys()):
    for j, output in enumerate(outputs):
        scatter = axs[i, j].scatter(
            df[df["log_name"] == log_name]["ps_percent"],
            df[df["log_name"] == log_name]["TCIPC_MaxTipDeflection"],
            c=df[df["log_name"] == log_name][output],
        )

        axs[i, j].set_xlabel("Peak shaving (-)")
        axs[i, j].set_ylabel("TCIPC max tip deflection (m)")
        axs[i, j].set_title(output)
        plt.colorbar(scatter, ax=axs[i, j])

plt.show()

In [ ]:
# Get an idea of the trade-off between AEP and tower clearance.
fig, ax = plt.subplots()

df.sort_values("aep", inplace=True)

sns.lineplot(
    data=df[df["TCIPC_MaxTipDeflection"] == 0.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
)
sns.lineplot(
    data=df[df["TCIPC_MaxTipDeflection"] == 20.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
    palette="deep",
)

## Data interpolation

In [ ]:
# Define the bounds of our design space for interpolation.
ps_min, ps_max = (
    df["ps_percent"].min(),
    df["ps_percent"].max(),
)
tip_min, tip_max = (
    df["TCIPC_MaxTipDeflection"].min(),
    df["TCIPC_MaxTipDeflection"].max(),
)

# Create a regular grid for interpolation to enable smooth contour plots.
n_points = 50
ps_grid = np.linspace(ps_min, ps_max, n_points)
tip_grid = np.linspace(tip_min, tip_max, n_points)
ps_percent_grid, tcipc_reference_grid = np.meshgrid(ps_grid, tip_grid)

# Interpolate each output variable on the grid for each log.
# We store the results in a nested dictionary for easy access when plotting.
interpolated_data = {}

for log_name in logs_to_load.keys():
    interpolated_data[log_name] = {}
    df_log = df[df["log_name"] == log_name]

    # Extract the independent variables as points for interpolation.
    points = df_log[["ps_percent", "TCIPC_MaxTipDeflection"]].values

    for output in outputs:
        # Extract the dependent variable values.
        values = df_log[output].values

        # Interpolate using linear method, which works well for scattered data.
        grid_values = griddata(
            points,
            values,
            (ps_percent_grid, tcipc_reference_grid),
            method="linear",
        )

        interpolated_data[log_name][output] = grid_values

print(f"Interpolated {len(outputs)} outputs for {len(logs_to_load)} datasets")
print(f"Grid shape: {ps_percent_grid.shape}")

In [ ]:
# Plot several outputs/objectives as a function of the design variables.
# Output variables to plot.
outputs = [
    "aep",
    "tower_clearance",
    "avg_pitch_travel",
    "TCIPC_amplitude_at_max_deflection",
    "DEL_RootMyb",
    "max_TwrBsMyt",
]
clims = [
    [70, 95],
    [10, 30],
    [0, 2],
    [0, 5],
    [15, 50],
    [150, 400],
]

fig, axs = plt.subplots(len(logs_to_load), len(outputs), figsize=(12, 5))

for i, log_name in enumerate(logs_to_load.keys()):
    for j, output in enumerate(outputs):
        levels = np.linspace(clims[j][0], clims[j][1], 9)
        scatter = axs[i, j].contourf(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=clims[j][0],
            vmax=clims[j][1],
        )

        axs[i, j].set_xlabel("Peak shaving (-)")
        axs[i, j].set_ylabel("TCIPC max tip deflection (m)")
        axs[i, j].set_title(output)
        plt.colorbar(scatter, ax=axs[i, j])


In [ ]:
# Investigate one plot in detail.
fig, ax = plt.subplots()

scatter = ax.contourf(
    ps_percent_grid,
    tcipc_reference_grid,
    interpolated_data["Free yaw"]["DEL_RootMyb"],
    levels=np.linspace(25, 35, 100),
    # levels=np.linspace(0.15, 1.5, 100),
    # levels=np.linspace(260, 420, 100),
)

# plt.title("DEL_RootMyb")
plt.colorbar(scatter)

## Optimization

In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.ctaea import CTAEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import Hypervolume
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator
from copy import deepcopy
from itertools import cycle

In [ ]:
class InterpolatorSet:
    """Builds interpolators for all numeric columns of a dataframe subset.

    Given a dataframe (typically filtered to one log_name), this creates a
    scipy interpolator for each numeric column, using two specified design
    variable columns as inputs.
    """

    def __init__(self, df, design_vars, method="linear"):
        xy = df[list(design_vars)].values

        Interpolator = (
            CloughTocher2DInterpolator if method == "cubic" else LinearNDInterpolator
        )

        # Build an interpolator for every numeric column that is not a
        # design variable.
        self._interpolators = {}
        for col in df.select_dtypes(include=[np.number]).columns:
            if col in design_vars:
                continue
            self._interpolators[col] = Interpolator(xy, df[col].values)

    def __getitem__(self, column):
        return self._interpolators[column]

In [ ]:
class OptimizationProblem:
    """Declarative specification of a multi-objective optimization problem.

    Objectives are specified as dicts:
        {"variable": str, "direction": "minimize" | "maximize", "label": str}

    Constraints are specified as dicts:
        {
            "name": str,           # Descriptive name for plotting.
            "variable": str,       # Column name from InterpolatorSet.
            "label": str,          # Axis label including units.
            "threshold": float,    # Pre-computed absolute threshold value.
            "direction": "below" | "above"  # "below" = must stay below threshold.
        }
    """

    def __init__(self, objectives, constraints, xl, xu):
        self.objectives = objectives
        self.constraints = constraints
        self.xl = np.array(xl)
        self.xu = np.array(xu)

    def to_pymoo(self, interp_set):
        """Build a pymoo ElementwiseProblem from this specification.

        Pymoo always minimizes, so objectives with direction="maximize" are
        negated internally.
        """
        objectives = self.objectives
        constraints = self.constraints
        xl = self.xl
        xu = self.xu

        # Build objective callables. Negate "maximize" objectives so pymoo
        # can minimize them.
        obj_callables = []
        for obj in objectives:
            interp = interp_set[obj["variable"]]
            sign = -1.0 if obj["direction"] == "maximize" else 1.0
            obj_callables.append(lambda x, y, _i=interp, _s=sign: _s * _i(x, y))

        # Build constraint callables. G <= 0 is feasible.
        con_callables = []
        for con in constraints:
            interp = interp_set[con["variable"]]
            threshold = con["threshold"]
            if con["direction"] == "below":
                # Feasible when current <= threshold.
                con_callables.append(
                    lambda x, y, _i=interp, _t=threshold: _i(x, y) - _t
                )
            else:
                # Feasible when current >= threshold.
                con_callables.append(
                    lambda x, y, _i=interp, _t=threshold: _t - _i(x, y)
                )

        class _Problem(ElementwiseProblem):
            def __init__(self):
                super().__init__(
                    n_var=2,
                    n_obj=len(objectives),
                    n_ieq_constr=len(constraints),
                    xl=xl,
                    xu=xu,
                )

            def _evaluate(self, x, out, *args, **kwargs):
                out["F"] = [f(*x) for f in obj_callables]
                out["G"] = [g(*x) for g in con_callables]

        return _Problem()

In [ ]:
class OptimizationStudy:
    """Runs multi-objective optimization across datasets and algorithms.

    Runs the OptimizationProblem on every log_name in the dataframe, using
    every algorithm in the provided dict. Also runs a baseline variant where
    the second design variable is fixed at its upper bound (TCIPC disabled),
    using the baseline_log dataset.
    """

    MARKERS = ["o", "s", "^", "D", "v", "P", "*", "X"]

    def __init__(
        self,
        df,
        problem,
        algorithms,
        termination,
        design_vars,
        baseline_log,
        interp_method="linear",
        seed=1,
        verbose=False,
    ):
        self.df = df
        self.problem = problem
        self.algorithms = algorithms
        self.termination = termination
        self.design_vars = design_vars
        self.baseline_log = baseline_log
        self.interp_method = interp_method
        self.seed = seed
        self.verbose = verbose

        self.log_names = df["log_name"].unique().tolist()
        self.results = {}
        self.interp_sets = {}

    def run(self):
        """Run all optimizations: baseline + each (log_name, algorithm)."""
        # Build interpolator sets for all datasets.
        for log_name in self.log_names:
            df_log = self.df[self.df["log_name"] == log_name]
            self.interp_sets[log_name] = InterpolatorSet(
                df_log, self.design_vars, method=self.interp_method
            )

        # Run baseline variant: TCIPC fixed at upper bound, only peak shaving varies.
        bl_problem = deepcopy(self.problem)
        bl_problem.xl[1] = bl_problem.xu[1]
        bl_interp = self.interp_sets[self.baseline_log]

        for algo_name, algo_factory in self.algorithms.items():
            result = minimize(
                bl_problem.to_pymoo(bl_interp),
                deepcopy(algo_factory),
                deepcopy(self.termination),
                seed=self.seed,
                save_history=True,
                verbose=self.verbose,
            )
            self.results[("Baseline", algo_name)] = result
            print(f"  Done: Baseline / {algo_name} -> {len(result.F)} solutions")

        # Run full optimizations for each dataset and algorithm.
        for log_name in self.log_names:
            interp_set = self.interp_sets[log_name]
            for algo_name, algo_factory in self.algorithms.items():
                result = minimize(
                    self.problem.to_pymoo(interp_set),
                    deepcopy(algo_factory),
                    deepcopy(self.termination),
                    seed=self.seed,
                    save_history=True,
                    verbose=self.verbose,
                )
                self.results[(log_name, algo_name)] = result
                print(f"  Done: {log_name} / {algo_name} -> {len(result.F)} solutions")

    @staticmethod
    def _calculate_hv_history(result):
        """Compute hypervolume at each generation from a pymoo result."""
        hist_F = []
        for algo in result.history:
            opt = algo.opt
            feas = np.where(opt.get("feasible"))[0]
            hist_F.append(opt.get("F")[feas])

        approx_ideal = result.F.min(axis=0)
        approx_nadir = result.F.max(axis=0)

        metric = Hypervolume(
            ref_point=approx_nadir,
            norm_ref_point=False,
            zero_to_one=False,
            ideal=approx_ideal,
            nadir=approx_nadir,
        )
        return [metric.do(_F) for _F in hist_F]

    def plot_convergence(self):
        """Plot hypervolume convergence for all optimization runs."""
        fig, ax = plt.subplots()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            hv = self._calculate_hv_history(result)
            ax.plot(
                hv,
                marker=next(markers),
                markersize=3,
                label=f"{log_name} / {algo_name}",
            )

        ax.set_xlabel("Generation")
        ax.set_ylabel("Hypervolume")
        ax.legend()

    def plot_pareto(self):
        """Plot Pareto front comparison for all runs (2 objectives).

        Objective values are converted back to the original direction by
        un-negating maximized objectives.
        """
        signs = np.array(
            [
                -1.0 if o["direction"] == "maximize" else 1.0
                for o in self.problem.objectives
            ]
        )

        fig, ax = plt.subplots()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            F = result.F * signs
            sort_idx = F[:, 0].argsort()
            F = F[sort_idx]
            ax.plot(
                F[:, 0],
                F[:, 1],
                marker=next(markers),
                markersize=4,
                label=f"{log_name} / {algo_name}",
            )

        ax.set_xlabel(self.problem.objectives[0]["label"])
        ax.set_ylabel(self.problem.objectives[1]["label"])
        ax.legend()

    def plot_design_space(self):
        """Plot design variable values for all Pareto-optimal points."""
        fig, ax = plt.subplots()
        markers = cycle(self.MARKERS)

        for (log_name, algo_name), result in self.results.items():
            ax.scatter(
                result.X[:, 0],
                result.X[:, 1],
                marker=next(markers),
                s=20,
                label=f"{log_name} / {algo_name}",
            )

        ax.set_xlim(self.problem.xl[0], self.problem.xu[0])
        ax.set_ylim(0, self.problem.xu[1])
        ax.set_xlabel(labels.get(self.design_vars[0], self.design_vars[0]))
        ax.set_ylabel(labels.get(self.design_vars[1], self.design_vars[1]))
        ax.legend()

    def plot_constraints(self):
        """Plot constraint satisfaction for Pareto-optimal solutions.

        Creates one subplot per constraint. Shows absolute values of the
        constrained variable for each solution, with a horizontal line at the
        threshold. This makes it easy to sanity-check the constraint values.
        """
        n_constraints = len(self.problem.constraints)
        fig, axs = plt.subplots(1, n_constraints, squeeze=False)

        markers = cycle(self.MARKERS)
        # Pre-assign a marker per result so it is consistent across subplots.
        result_markers = {key: next(markers) for key in self.results}

        for j, con in enumerate(self.problem.constraints):
            ax = axs[0, j]
            threshold = con["threshold"]

            for (log_name, algo_name), result in self.results.items():
                # Recover the absolute value from the pymoo G output.
                # For "below": G = current - threshold, so current = G + threshold.
                # For "above": G = threshold - current, so current = threshold - G.
                if con["direction"] == "below":
                    values = result.G[:, j] + threshold
                else:
                    values = threshold - result.G[:, j]

                ax.scatter(
                    np.arange(len(values)),
                    values,
                    marker=result_markers[(log_name, algo_name)],
                    s=15,
                    label=f"{log_name} / {algo_name}",
                )

            ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
            ax.set_xlabel("Solution index")
            ax.set_ylabel(con["label"])
            ax.set_title(con["name"])
            ax.legend()

        fig.tight_layout()

In [ ]:
# Compute baseline constraint thresholds from the "Free yaw" dataset at the
# baseline operating point (ps_percent=0.8, TCIPC_MaxTipDeflection=20.0).
# This is the turbine without TCIPC, so all constraints are relative to this.
BASELINE_LOG = "Free yaw"
BASELINE_POINT = (0.8, 20.0)
DESIGN_VARS = ("ps_percent", "TCIPC_MaxTipDeflection")

baseline_interp = InterpolatorSet(
    df[df["log_name"] == BASELINE_LOG], DESIGN_VARS, method="linear"
)

baseline_DEL_RootMyb = float(baseline_interp["DEL_RootMyb"](*BASELINE_POINT))
baseline_max_TwrBsMyt = float(baseline_interp["max_TwrBsMyt"](*BASELINE_POINT))
baseline_tower_clearance = float(baseline_interp["tower_clearance"](*BASELINE_POINT))

print(f"Baseline ({BASELINE_LOG} at {BASELINE_POINT}):")
print(f"  DEL_RootMyb  = {baseline_DEL_RootMyb:.2f} MNm")
print(f"  max_TwrBsMyt = {baseline_max_TwrBsMyt:.2f} MNm")
print(f"  tower_clearance = {baseline_tower_clearance:.2f} m")

In [ ]:
# Define the optimization problem.
problem = OptimizationProblem(
    objectives=[
        # {
        #     "variable": "avg_pitch_travel",
        #     "direction": "minimize",
        #     "label": "Pitch travel (-)",
        # },
        {
            "variable": "aep",
            "direction": "maximize",
            "label": "AEP (GWh)",
        },
        {
            "variable": "tower_clearance",
            "direction": "maximize",
            "label": "Tower clearance (m)",
        }
    ],
    constraints=[
        # Blade root DEL must not exceed the baseline value.
        {
            "name": "Blade root DEL",
            "variable": "DEL_RootMyb",
            "label": "DEL root Myb (MNm)",
            "threshold": baseline_DEL_RootMyb,
            "direction": "below",
        },
        # Tower base moment must not exceed the baseline value.
        {
            "name": "Tower base moment",
            "variable": "max_TwrBsMyt",
            "label": "Max tower base Myt (MNm)",
            "threshold": baseline_max_TwrBsMyt,
            "direction": "below",
        },
        # {
        #     "name": "Tower clearance",
        #     "variable": "tower_clearance",
        #     "label": "Tower clearance (m)",
        #     "threshold": baseline_tower_clearance,
        #     "direction": "above",
        # },
    ],
    xl=[0.5, 0.0],
    xu=[1.0, 20.0],
)

# Specify the algorithms to compare.
ref_dirs = get_reference_directions("uniform", 2, n_partitions=100)

algorithms = {
    "NSGA2": NSGA2(
        pop_size=100,
        n_offsprings=20,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=15),
        mutation=PM(eta=15),
        eliminate_duplicates=True,
    ),
    # "MOEAD": MOEAD(
    #     ref_dirs,
    #     n_neighbors=50,
    #     prob_neighbor_mating=0.7,
    # ),
    # "CTAEA": CTAEA(
    #     ref_dirs,
    #     sampling=FloatRandomSampling(),
    #     crossover=SBX(prob=0.9, eta=5),
    #     mutation=PM(eta=10),
    #     eliminate_duplicates=True,
    # ),
}

termination = get_termination("n_gen", 100)

# Run the study across all datasets and algorithms.
study = OptimizationStudy(
    df,
    problem=problem,
    algorithms=algorithms,
    termination=termination,
    design_vars=DESIGN_VARS,
    baseline_log=BASELINE_LOG,
    interp_method="linear",
)
study.run()

In [ ]:
# Hypervolume convergence comparison across all runs.
study.plot_convergence()
plt.show()

In [ ]:
# Pareto front comparison across all runs.
study.plot_pareto()
plt.show()

In [ ]:
# Design space comparison across all runs.
study.plot_design_space()
plt.show()

In [ ]:
# Constraint satisfaction visualization.
study.plot_constraints()
plt.show()